<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/06_GES_Aware_Genomic_RAG_Cell_7B5_Stage_7C_Execution_Authorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive when running in Google Colab.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Google Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root does not exist: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder name is exact.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Frozen identities, authorization boundary, and output locations

In [2]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

CELL_ID = '7B5'
STAGE = '7B'
PACKAGE_VERSION = '1.0.0'
NOTEBOOK_NAME = '06_GES_Aware_Genomic_RAG_Cell_7B5_Stage_7C_Execution_Authorization.ipynb'
FROZEN_SEED = 20260722

EXPECTED_CELL_7B4_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_'
    'SIMILARITY_TOP20_CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_'
    'BLINDED_ALIASES_PROMPTS_STRICT_RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_'
    'GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_FROZEN_CHECKSUM_PROTECTED_'
    'NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_LLM_'
    'ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7B4 = OrderedDict([
    ('embedding_retrieval', {
        'filename': 'cell_7b4_embedding_retrieval_configuration_v1.json',
        'sha256': 'ab209e48b025652e5ddbb79891225334ffc51de62f157b430c21aef30865b807',
    }),
    ('quality_reranking', {
        'filename': 'cell_7b4_quality_reranking_configuration_v1.json',
        'sha256': '35c3871db6dd6bad9436e7202a67b064cc98beee06d19894e409d42f7ca006fb',
    }),
    ('llm_prompt_response', {
        'filename': 'cell_7b4_llm_prompt_response_configuration_v1.json',
        'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
    }),
    ('runtime_determinism', {
        'filename': 'cell_7b4_runtime_and_determinism_configuration_v1.json',
        'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
    }),
    ('condition_aliases', {
        'filename': 'cell_7b4_condition_alias_inventory_v1.csv',
        'sha256': '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9',
    }),
    ('requirements_lock', {
        'filename': 'cell_7b4_execution_requirements_lock_v1.txt',
        'sha256': '1a894f7ba976d00563325cc3324ff91708b5699e1c618e3674502fe586828a91',
    }),
    ('qc', {
        'filename': 'cell_7b4_configuration_freeze_qc_v1.json',
        'sha256': '01d6ef836c25d49b9df32d2739d1d532a1f205be0d83c6227eb65070d9f9ced4',
    }),
    ('manifest', {
        'filename': 'cell_7b4_configuration_freeze_manifest_v1.json',
        'sha256': '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe',
    }),
])

# Only the score-blind inputs needed by Stage 7C Cell 7C0 are authorized here.
EXPECTED_CELL_7B3_EXECUTION_INPUTS = OrderedDict([
    ('semantic_corpus', {
        'filename': 'cell_7b3_score_blind_semantic_corpus_v1.parquet',
        'sha256': '2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399',
        'expected_rows': 100_920,
    }),
    ('primary_questions', {
        'filename': 'cell_7b3_primary_question_set_v1.csv',
        'sha256': 'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df',
        'expected_rows': 80,
    }),
])

AUTH_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b5_execution_authorization_v1'
QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / 'cell_7b5_execution_authorization_v1'

OUTPUTS = OrderedDict([
    ('authorization', AUTH_DIR / 'cell_7b5_stage7c_cell7c0_execution_authorization_v1.json'),
    ('input_inventory', AUTH_DIR / 'cell_7b5_authorized_execution_input_inventory_v1.csv'),
    ('qc', QC_DIR / 'cell_7b5_execution_authorization_qc_v1.json'),
    ('manifest', AUTH_DIR / 'cell_7b5_execution_authorization_manifest_v1.json'),
])

for directory in (AUTH_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS['manifest'].exists():
    raise FileExistsError(
        f"Cell 7B5 manifest already exists: {OUTPUTS['manifest']}\n"
        'Fail-closed overwrite protection is active. Do not overwrite a frozen authorization package.'
    )

print(f'Authorization directory: {AUTH_DIR}')
print(f'QC directory           : {QC_DIR}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7b5_execution_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7b5_execution_authorization_v1


## 2. Checksum helpers and exact artifact discovery

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip().lower()
    if re.fullmatch(r'[0-9a-f]{64}', token) is None:
        raise ValueError(f'Invalid SHA-256 sidecar format: {path}')
    return token


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def locate_exact_hash(filename: str, expected_hash: str) -> Path:
    candidates = sorted(ROOT.rglob(filename))
    matches = [
        path for path in candidates
        if path.is_file() and sha256_file(path) == expected_hash
    ]
    if len(matches) != 1:
        details = '\n'.join(str(path) for path in candidates) or '<none>'
        raise RuntimeError(
            f'Expected exactly one checksum-matching artifact for {filename}; '
            f'found {len(matches)}.\nCandidates:\n{details}'
        )
    return matches[0]


def csv_data_row_count(path: Path) -> int:
    with path.open('rb') as handle:
        line_count = sum(1 for _ in handle)
    return max(0, line_count - 1)


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def stable_write_json(path: Path, payload: Any) -> str:
    data = (
        json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + '\n'
    ).encode('utf-8')
    atomic_write_bytes(path, data)
    return sha256_file(path)


def stable_write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            extrasaction='raise',
            lineterminator='\n',
        )
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return sha256_file(path)


def stable_write_text(path: Path, text: str) -> str:
    atomic_write_bytes(path, text.encode('utf-8'))
    return sha256_file(path)


def write_sidecar(path: Path) -> str:
    digest = sha256_file(path)
    sc = sidecar_path(path)
    stable_write_text(sc, f'{digest}  {path.name}\n')
    return sha256_file(sc)

print('Checksum and stable-write helpers loaded.')

Checksum and stable-write helpers loaded.


## 3. Reverify the complete Cell 7B4 configuration freeze

In [4]:
cell_7b4_paths: OrderedDict[str, Path] = OrderedDict()
cell_7b4_inventory: list[dict[str, Any]] = []

for artifact_id, spec in EXPECTED_CELL_7B4.items():
    path = locate_exact_hash(spec['filename'], spec['sha256'])
    if not sidecar_is_valid(path):
        raise AssertionError(f'Missing or invalid SHA-256 sidecar: {path}')
    cell_7b4_paths[artifact_id] = path
    cell_7b4_inventory.append({
        'source_cell': '7B4',
        'artifact_id': artifact_id,
        'filename': path.name,
        'path': str(path),
        'sha256': sha256_file(path),
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
        'authorized_for_cell_7c0': artifact_id in {
            'embedding_retrieval',
            'runtime_determinism',
            'requirements_lock',
            'manifest',
        },
    })

embedding_config = json.loads(cell_7b4_paths['embedding_retrieval'].read_text(encoding='utf-8'))
reranking_config = json.loads(cell_7b4_paths['quality_reranking'].read_text(encoding='utf-8'))
llm_config = json.loads(cell_7b4_paths['llm_prompt_response'].read_text(encoding='utf-8'))
runtime_config = json.loads(cell_7b4_paths['runtime_determinism'].read_text(encoding='utf-8'))
cell_7b4_qc = json.loads(cell_7b4_paths['qc'].read_text(encoding='utf-8'))
cell_7b4_manifest = json.loads(cell_7b4_paths['manifest'].read_text(encoding='utf-8'))

with cell_7b4_paths['condition_aliases'].open('r', encoding='utf-8', newline='') as handle:
    condition_aliases = list(csv.DictReader(handle))

cell_7b4_terminal_decision = str(cell_7b4_manifest.get('terminal_decision', ''))
scientific_boundary = cell_7b4_manifest.get('scientific_boundary', {})

cell_7b4_checks = OrderedDict([
    ('eight_cell_7b4_artifacts_found', len(cell_7b4_paths) == 8),
    ('eight_cell_7b4_hashes_exact', all(
        sha256_file(cell_7b4_paths[key]) == spec['sha256']
        for key, spec in EXPECTED_CELL_7B4.items()
    )),
    ('eight_cell_7b4_sidecars_valid', all(sidecar_is_valid(path) for path in cell_7b4_paths.values())),
    ('cell_7b4_terminal_decision_exact', cell_7b4_terminal_decision == EXPECTED_CELL_7B4_DECISION),
    ('cell_7b4_qc_zero_failures', int(cell_7b4_qc.get('failed_checks', -1)) == 0),
    ('cell_7b4_manifest_did_not_autoauthorize_execution', cell_7b4_manifest.get('next_authorized_cell') is None),
    ('cell_7b4_requires_separate_authorization', 'separate fail-closed execution-authorization' in str(cell_7b4_manifest.get('next_required_action', '')).lower()),
    ('cell_7b4_scientific_boundary_present', isinstance(scientific_boundary, dict) and len(scientific_boundary) > 0),
    ('cell_7b4_all_scientific_operations_false', all(value is False for value in scientific_boundary.values())),
    ('embedding_model_exact', embedding_config['embedding']['model_id'] == 'NeuML/pubmedbert-base-embeddings'),
    ('embedding_revision_exact', embedding_config['embedding']['revision'] == 'b79526d6ef3645e0df4530322e266f24c829f5ef'),
    ('embedding_dimension_768', embedding_config['embedding']['expected_embedding_dimension'] == 768),
    ('embedding_l2_normalized', embedding_config['embedding']['normalize_embeddings_l2'] is True),
    ('exact_faiss_index_flatip', embedding_config['semantic_similarity']['faiss_index'] == 'IndexFlatIP'),
    ('approximate_search_disabled', embedding_config['semantic_similarity']['approximate_nearest_neighbor'] is False),
    ('common_semantic_top20', embedding_config['candidate_pool']['semantic_candidate_pool_k'] == 20 and embedding_config['candidate_pool']['same_top20_pool_for_all_conditions'] is True),
    ('final_top5_frozen_but_not_authorized_in_7c0', embedding_config['candidate_pool']['final_context_k'] == 5),
    ('hard_exclusion_prohibited', embedding_config['candidate_pool']['hard_evidence_exclusion'] is False),
    ('rrf_weights_frozen', reranking_config['weighted_reciprocal_rank_fusion']['semantic_weight'] == 0.75 and reranking_config['weighted_reciprocal_rank_fusion']['quality_weight'] == 0.25),
    ('rrf_constant_60', reranking_config['weighted_reciprocal_rank_fusion']['rrf_constant'] == 60),
    ('six_blinded_aliases', len(condition_aliases) == 6 and len({row['blinded_alias'] for row in condition_aliases}) == 6),
    ('llm_snapshot_frozen_but_not_authorized_in_7c0', llm_config['llm']['model'] == 'gpt-4.1-mini-2025-04-14'),
    ('llm_temperature_zero', llm_config['generation']['temperature'] == 0.0),
    ('llm_three_repetitions', llm_config['generation']['repetitions_per_question_condition'] == 3),
    ('runtime_seed_exact', runtime_config['deterministic_controls']['global_seed'] == FROZEN_SEED),
    ('runtime_exact_index_policy', runtime_config['deterministic_controls']['faiss_index_type'] == 'IndexFlatIP exact search'),
    ('runtime_scores_separate', runtime_config['leakage_and_invariance_controls']['cell_7a3_scores_separate_from_score_blind_package'] is True),
    ('runtime_answer_keys_unavailable', runtime_config['leakage_and_invariance_controls']['answer_keys_unavailable_to_retrieval_and_generation_code'] is True),
])

failed_cell_7b4 = [name for name, passed in cell_7b4_checks.items() if not bool(passed)]
if failed_cell_7b4:
    raise RuntimeError('Cell 7B4 reverification failed:\n- ' + '\n- '.join(failed_cell_7b4))

print(f'Cell 7B4 exact artifacts reverified: {len(cell_7b4_paths)}/8')
print('Cell 7B4 terminal PASS verified   : YES')
print('Cell 7B4 scientific execution    : NONE')

Cell 7B4 exact artifacts reverified: 8/8
Cell 7B4 terminal PASS verified   : YES
Cell 7B4 scientific execution    : NONE


## 4. Reverify only the score-blind inputs authorized for Cell 7C0

In [5]:
execution_input_paths: OrderedDict[str, Path] = OrderedDict()
execution_input_inventory: list[dict[str, Any]] = []

for artifact_id, spec in EXPECTED_CELL_7B3_EXECUTION_INPUTS.items():
    path = locate_exact_hash(spec['filename'], spec['sha256'])
    if not sidecar_is_valid(path):
        raise AssertionError(f'Missing or invalid SHA-256 sidecar: {path}')
    execution_input_paths[artifact_id] = path
    row_count = csv_data_row_count(path) if path.suffix.lower() == '.csv' else spec['expected_rows']
    execution_input_inventory.append({
        'source_cell': '7B3',
        'artifact_id': artifact_id,
        'filename': path.name,
        'path': str(path),
        'sha256': sha256_file(path),
        'bytes': int(path.stat().st_size),
        'expected_rows': int(spec['expected_rows']),
        'observed_structural_rows': int(row_count),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
        'authorized_for_cell_7c0': True,
    })

# The Cell 7B4 manifest must independently carry the same exact upstream identities.
manifest_upstream_rows = cell_7b4_manifest.get('upstream_cell_7b3', {}).get('artifacts', [])
manifest_upstream_by_filename = {
    str(row.get('filename')): row for row in manifest_upstream_rows
    if isinstance(row, dict)
}

input_checks = OrderedDict([
    ('two_score_blind_inputs_found', len(execution_input_paths) == 2),
    ('semantic_corpus_hash_exact', sha256_file(execution_input_paths['semantic_corpus']) == EXPECTED_CELL_7B3_EXECUTION_INPUTS['semantic_corpus']['sha256']),
    ('primary_questions_hash_exact', sha256_file(execution_input_paths['primary_questions']) == EXPECTED_CELL_7B3_EXECUTION_INPUTS['primary_questions']['sha256']),
    ('both_input_sidecars_valid', all(sidecar_is_valid(path) for path in execution_input_paths.values())),
    ('primary_questions_80_rows', csv_data_row_count(execution_input_paths['primary_questions']) == 80),
    ('semantic_corpus_present_in_cell_7b4_manifest', EXPECTED_CELL_7B3_EXECUTION_INPUTS['semantic_corpus']['filename'] in manifest_upstream_by_filename),
    ('primary_questions_present_in_cell_7b4_manifest', EXPECTED_CELL_7B3_EXECUTION_INPUTS['primary_questions']['filename'] in manifest_upstream_by_filename),
    ('semantic_corpus_manifest_hash_exact', str(manifest_upstream_by_filename.get(EXPECTED_CELL_7B3_EXECUTION_INPUTS['semantic_corpus']['filename'], {}).get('sha256', '')) == EXPECTED_CELL_7B3_EXECUTION_INPUTS['semantic_corpus']['sha256']),
    ('primary_questions_manifest_hash_exact', str(manifest_upstream_by_filename.get(EXPECTED_CELL_7B3_EXECUTION_INPUTS['primary_questions']['filename'], {}).get('sha256', '')) == EXPECTED_CELL_7B3_EXECUTION_INPUTS['primary_questions']['sha256']),
    ('answer_key_not_opened', True),
    ('cell_7a3_score_table_not_opened', True),
])

failed_inputs = [name for name, passed in input_checks.items() if not bool(passed)]
if failed_inputs:
    raise RuntimeError('Authorized-input reverification failed:\n- ' + '\n- '.join(failed_inputs))

print('Semantic corpus exact hash verified : YES')
print('Primary question set verified       : 80 rows')
print('Answer-key outcomes inspected       : NO')
print('Cell 7A3 score table opened          : NO')

Semantic corpus exact hash verified : YES
Primary question set verified       : 80 rows
Answer-key outcomes inspected       : NO
Cell 7A3 score table opened          : NO


## 5. Freeze the narrow Stage 7C Cell 7C0 execution authorization

In [6]:
AUTHORIZED_CELL_7C0_OUTPUT_SPEC = OrderedDict([
    ('model_artifact_inventory', 'cell_7c0_embedding_model_artifact_inventory_v1.json'),
    ('corpus_embeddings', 'cell_7c0_semantic_corpus_embeddings_float32_l2_v1.npy'),
    ('corpus_embedding_identity', 'cell_7c0_semantic_corpus_embedding_identity_v1.parquet'),
    ('question_embeddings', 'cell_7c0_primary_question_embeddings_float32_l2_v1.npy'),
    ('question_embedding_identity', 'cell_7c0_primary_question_embedding_identity_v1.csv'),
    ('faiss_index', 'cell_7c0_semantic_corpus_indexflatip_v1.faiss'),
    ('semantic_top20_pool', 'cell_7c0_common_semantic_top20_candidate_pool_v1.parquet'),
    ('execution_report', 'cell_7c0_embedding_and_semantic_retrieval_report_v1.json'),
    ('qc', 'cell_7c0_embedding_and_semantic_retrieval_qc_v1.json'),
    ('manifest', 'cell_7c0_embedding_and_semantic_retrieval_manifest_v1.json'),
])

created_utc = datetime.now(timezone.utc).isoformat()

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': created_utc,
    'authorization_type': 'fail_closed_scientific_execution_authorization',
    'authorization_decision': 'AUTHORIZE_STAGE7C_CELL7C0_EMBEDDING_AND_SEMANTIC_RETRIEVAL_EXECUTION_ONLY',
    'authorized_cell': {
        'stage': '7C',
        'cell_id': '7C0',
        'title': 'Frozen embedding generation, exact FAISS indexing, and common semantic top-20 retrieval',
        'authorized_once': True,
        'overwrite_existing_outputs': False,
    },
    'authorized_operations': [
        'Install or verify the exact Cell 7B4 frozen runtime requirements.',
        'Load only the frozen score-blind semantic corpus and the 80 frozen primary questions.',
        'Apply the exact Cell 7B4 text-normalization implementation.',
        'Download and load NeuML/pubmedbert-base-embeddings at revision b79526d6ef3645e0df4530322e266f24c829f5ef.',
        'Generate float32 L2-normalized corpus embeddings for all 100,920 semantic-corpus rows.',
        'Generate float32 L2-normalized question embeddings for all 80 primary questions.',
        'Construct one exact FAISS IndexFlatIP index over the frozen corpus embeddings.',
        'Retrieve one common semantic top-20 candidate pool per question using exact normalized inner product.',
        'Freeze identities, vectors, index, top-20 results, runtime lineage, QC, manifest, and SHA-256 sidecars.',
    ],
    'prohibited_operations': [
        'Load or inspect Cell 7A3 Full-GES, no-star-GES, or combined-metadata score values.',
        'Construct quality ranks or apply any quality-aware reranking.',
        'Apply the 0.75/0.25 RRF rule.',
        'Select or materialize the final top-5 context.',
        'Duplicate, mutate, filter, or hard-exclude records from the common semantic top-20 pool.',
        'Materialize condition-specific prompts.',
        'Call any LLM or external answering service.',
        'Load or inspect structured answer-key outcomes.',
        'Adjudicate answers or calculate retrieval, answer, calibration, or RAG performance metrics.',
        'Tune the embedding model, revision, normalization, top-k, similarity, RRF weights, prompts, endpoints, or questions.',
    ],
    'frozen_execution_design': {
        'embedding_model': embedding_config['embedding']['model_id'],
        'embedding_revision': embedding_config['embedding']['revision'],
        'embedding_dimension': embedding_config['embedding']['expected_embedding_dimension'],
        'max_sequence_length_tokens': embedding_config['embedding']['max_sequence_length_tokens'],
        'output_dtype': embedding_config['embedding']['output_dtype'],
        'l2_normalization': embedding_config['embedding']['normalize_embeddings_l2'],
        'text_normalization_reference_sha256': embedding_config['text_normalization']['reference_implementation_sha256'],
        'similarity': embedding_config['semantic_similarity']['metric'],
        'faiss_index': embedding_config['semantic_similarity']['faiss_index'],
        'approximate_search': embedding_config['semantic_similarity']['approximate_nearest_neighbor'],
        'semantic_candidate_pool_k': embedding_config['candidate_pool']['semantic_candidate_pool_k'],
        'common_pool_for_all_conditions': embedding_config['candidate_pool']['same_top20_pool_for_all_conditions'],
        'hard_evidence_exclusion': embedding_config['candidate_pool']['hard_evidence_exclusion'],
        'global_seed': runtime_config['deterministic_controls']['global_seed'],
        'corpus_sort_before_embedding': runtime_config['deterministic_controls']['corpus_sort_before_embedding'],
        'question_sort_before_execution': runtime_config['deterministic_controls']['question_sort_before_execution'],
    },
    'authorized_inputs': {
        'cell_7b4_manifest': {
            'path': str(cell_7b4_paths['manifest']),
            'sha256': sha256_file(cell_7b4_paths['manifest']),
        },
        'embedding_retrieval_configuration': {
            'path': str(cell_7b4_paths['embedding_retrieval']),
            'sha256': sha256_file(cell_7b4_paths['embedding_retrieval']),
        },
        'runtime_determinism_configuration': {
            'path': str(cell_7b4_paths['runtime_determinism']),
            'sha256': sha256_file(cell_7b4_paths['runtime_determinism']),
        },
        'requirements_lock': {
            'path': str(cell_7b4_paths['requirements_lock']),
            'sha256': sha256_file(cell_7b4_paths['requirements_lock']),
        },
        'semantic_corpus': {
            'path': str(execution_input_paths['semantic_corpus']),
            'sha256': sha256_file(execution_input_paths['semantic_corpus']),
            'expected_rows': 100_920,
        },
        'primary_questions': {
            'path': str(execution_input_paths['primary_questions']),
            'sha256': sha256_file(execution_input_paths['primary_questions']),
            'expected_rows': 80,
        },
    },
    'required_cell_7c0_outputs': AUTHORIZED_CELL_7C0_OUTPUT_SPEC,
    'candidate_pool_required_schema': [
        'question_id',
        'semantic_rank',
        'semantic_score',
        'corpus_row_index',
        'packet_id',
        'rcv_accession',
    ],
    'candidate_pool_required_rows': 80 * 20,
    'scientific_claim_boundary': (
        'This authorization does not establish retrieval quality, RAG benefit, answer accuracy, '
        'clinical validity, clinical utility, or safety. Cell 7C0 is limited to frozen embedding '
        'and common semantic-candidate-pool materialization.'
    ),
}

print('Cell 7C0 authorization payload constructed in memory.')
print('Scientific execution performed in Cell 7B5: NO')

Cell 7C0 authorization payload constructed in memory.
Scientific execution performed in Cell 7B5: NO


## 6. QC, checksum-protected authorization materialization, and final boundary

In [7]:
immutable_paths = OrderedDict()
immutable_paths.update((f'7b4_{key}', path) for key, path in cell_7b4_paths.items())
immutable_paths.update((f'7b3_{key}', path) for key, path in execution_input_paths.items())
immutable_hashes_before = OrderedDict((key, sha256_file(path)) for key, path in immutable_paths.items())

all_input_rows = cell_7b4_inventory + execution_input_inventory

prewrite_checks = OrderedDict()
prewrite_checks.update(cell_7b4_checks)
prewrite_checks.update(input_checks)
prewrite_checks.update(OrderedDict([
    ('authorization_cell_is_7c0', authorization_payload['authorized_cell']['cell_id'] == '7C0'),
    ('authorization_stage_is_7c', authorization_payload['authorized_cell']['stage'] == '7C'),
    ('authorization_once', authorization_payload['authorized_cell']['authorized_once'] is True),
    ('overwrite_prohibited', authorization_payload['authorized_cell']['overwrite_existing_outputs'] is False),
    ('authorization_exact_model', authorization_payload['frozen_execution_design']['embedding_model'] == 'NeuML/pubmedbert-base-embeddings'),
    ('authorization_exact_revision', authorization_payload['frozen_execution_design']['embedding_revision'] == 'b79526d6ef3645e0df4530322e266f24c829f5ef'),
    ('authorization_dimension_768', authorization_payload['frozen_execution_design']['embedding_dimension'] == 768),
    ('authorization_l2_true', authorization_payload['frozen_execution_design']['l2_normalization'] is True),
    ('authorization_indexflatip', authorization_payload['frozen_execution_design']['faiss_index'] == 'IndexFlatIP'),
    ('authorization_top20', authorization_payload['frozen_execution_design']['semantic_candidate_pool_k'] == 20),
    ('authorization_common_pool', authorization_payload['frozen_execution_design']['common_pool_for_all_conditions'] is True),
    ('authorization_hard_exclusion_false', authorization_payload['frozen_execution_design']['hard_evidence_exclusion'] is False),
    ('candidate_pool_expected_1600_rows', authorization_payload['candidate_pool_required_rows'] == 1_600),
    ('required_output_count_10', len(AUTHORIZED_CELL_7C0_OUTPUT_SPEC) == 10),
    ('score_loading_explicitly_prohibited', any('Cell 7A3' in item for item in authorization_payload['prohibited_operations'])),
    ('reranking_explicitly_prohibited', any('reranking' in item.lower() for item in authorization_payload['prohibited_operations'])),
    ('top5_explicitly_prohibited', any('top-5' in item.lower() for item in authorization_payload['prohibited_operations'])),
    ('llm_explicitly_prohibited', any('llm' in item.lower() for item in authorization_payload['prohibited_operations'])),
    ('answer_key_explicitly_prohibited', any('answer-key' in item.lower() for item in authorization_payload['prohibited_operations'])),
    ('metrics_explicitly_prohibited', any('metrics' in item.lower() for item in authorization_payload['prohibited_operations'])),
    ('cell_7b5_generated_no_embeddings', True),
    ('cell_7b5_built_no_faiss_index', True),
    ('cell_7b5_executed_no_retrieval', True),
    ('cell_7b5_loaded_no_scores', True),
    ('cell_7b5_materialized_no_prompts', True),
    ('cell_7b5_called_no_llm', True),
    ('cell_7b5_inspected_no_answer_key_outcomes', True),
    ('cell_7b5_calculated_no_rag_metrics', True),
]))

failed_prewrite = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed_prewrite:
    raise RuntimeError('Cell 7B5 prewrite QC failed:\n- ' + '\n- '.join(failed_prewrite))

stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

stable_write_csv(
    OUTPUTS['input_inventory'],
    all_input_rows,
    fieldnames=[
        'source_cell',
        'artifact_id',
        'filename',
        'path',
        'sha256',
        'bytes',
        'expected_rows',
        'observed_structural_rows',
        'sidecar_path',
        'sidecar_valid',
        'authorized_for_cell_7c0',
    ],
)
write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7B5_CELL7B4_CONFIGURATION_AND_SCORE_BLIND_EXECUTION_INPUTS_'
    'REVERIFIED_STAGE7C_CELL7C0_EMBEDDING_EXACT_FAISS_INDEXFLATIP_AND_COMMON_'
    'SEMANTIC_TOP20_RETRIEVAL_EXECUTION_ONLY_EXPLICITLY_AUTHORIZED_CHECKSUM_'
    'PROTECTED_NO_CELL7A3_SCORES_QUALITY_RERANKING_RRF_TOP5_PROMPTS_LLM_'
    'ANSWER_KEY_OUTCOME_INSPECTION_ADJUDICATION_OR_RAG_METRICS'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': created_utc,
    'scope': 'stage7c_execution_authorization_only',
    'checks': [
        {'check': name, 'passed': bool(passed)}
        for name, passed in prewrite_checks.items()
    ],
    'passed_checks': int(sum(bool(value) for value in prewrite_checks.values())),
    'failed_checks': int(sum(not bool(value) for value in prewrite_checks.values())),
    'total_checks': int(len(prewrite_checks)),
    'scientific_operations': {
        'cell_7a3_scores_loaded': False,
        'answer_key_outcomes_inspected': False,
        'embedding_model_downloaded': False,
        'embeddings_generated': False,
        'faiss_index_constructed': False,
        'semantic_retrieval_executed': False,
        'quality_reranking_applied': False,
        'final_top5_materialized': False,
        'prompts_materialized': False,
        'llm_called': False,
        'adjudication_performed': False,
        'rag_metrics_calculated': False,
    },
    'decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

output_records = OrderedDict()
for key, path in OUTPUTS.items():
    if key == 'manifest':
        continue
    if not sidecar_is_valid(path):
        raise AssertionError(f'Output sidecar verification failed: {path}')
    output_records[key] = {
        'path': str(path),
        'sha256': sha256_file(path),
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_sha256': sha256_file(sidecar_path(path)),
    }

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'notebook_name': NOTEBOOK_NAME,
    'created_utc': created_utc,
    'purpose': (
        'Reverify the exact Cell 7B4 configuration freeze and issue a narrow, '
        'fail-closed authorization for Stage 7C Cell 7C0 embedding generation, '
        'exact FAISS indexing, and common semantic top-20 retrieval only.'
    ),
    'upstream_cell_7b4': {
        'manifest_path': str(cell_7b4_paths['manifest']),
        'manifest_sha256': sha256_file(cell_7b4_paths['manifest']),
        'terminal_decision': cell_7b4_terminal_decision,
        'artifacts': cell_7b4_inventory,
    },
    'authorized_score_blind_inputs': execution_input_inventory,
    'output_artifacts': output_records,
    'authorization': {
        'path': str(OUTPUTS['authorization']),
        'sha256': sha256_file(OUTPUTS['authorization']),
        'decision': authorization_payload['authorization_decision'],
    },
    'qc': {
        'path': str(OUTPUTS['qc']),
        'sha256': sha256_file(OUTPUTS['qc']),
        'passed_checks': qc_payload['passed_checks'],
        'failed_checks': qc_payload['failed_checks'],
        'total_checks': qc_payload['total_checks'],
    },
    'scientific_boundary': qc_payload['scientific_operations'],
    'terminal_decision': terminal_decision,
    'next_authorized_cell': {
        'stage': '7C',
        'cell_id': '7C0',
        'scope': 'embedding_generation_exact_faiss_index_and_common_semantic_top20_retrieval_only',
        'may_load_cell_7a3_scores': False,
        'may_apply_quality_reranking': False,
        'may_materialize_final_top5': False,
        'may_materialize_prompts': False,
        'may_call_llm': False,
        'may_open_answer_keys': False,
        'may_calculate_rag_metrics': False,
    },
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback checks.
authorization_readback = json.loads(OUTPUTS['authorization'].read_text(encoding='utf-8'))
qc_readback = json.loads(OUTPUTS['qc'].read_text(encoding='utf-8'))
manifest_readback = json.loads(OUTPUTS['manifest'].read_text(encoding='utf-8'))

readback_checks = OrderedDict([
    ('authorization_sidecar_valid', sidecar_is_valid(OUTPUTS['authorization'])),
    ('input_inventory_sidecar_valid', sidecar_is_valid(OUTPUTS['input_inventory'])),
    ('qc_sidecar_valid', sidecar_is_valid(OUTPUTS['qc'])),
    ('manifest_sidecar_valid', sidecar_is_valid(OUTPUTS['manifest'])),
    ('authorization_decision_readback', authorization_readback['authorization_decision'] == 'AUTHORIZE_STAGE7C_CELL7C0_EMBEDDING_AND_SEMANTIC_RETRIEVAL_EXECUTION_ONLY'),
    ('authorization_cell_readback_7c0', authorization_readback['authorized_cell']['cell_id'] == '7C0'),
    ('candidate_rows_readback_1600', authorization_readback['candidate_pool_required_rows'] == 1_600),
    ('qc_zero_failures_readback', qc_readback['failed_checks'] == 0),
    ('qc_decision_readback', qc_readback['decision'] == terminal_decision),
    ('manifest_decision_readback', manifest_readback['terminal_decision'] == terminal_decision),
    ('manifest_authorizes_only_7c0', manifest_readback['next_authorized_cell']['cell_id'] == '7C0'),
    ('manifest_7c0_cannot_load_scores', manifest_readback['next_authorized_cell']['may_load_cell_7a3_scores'] is False),
    ('manifest_7c0_cannot_rerank', manifest_readback['next_authorized_cell']['may_apply_quality_reranking'] is False),
    ('manifest_7c0_cannot_call_llm', manifest_readback['next_authorized_cell']['may_call_llm'] is False),
    ('all_cell_7b5_scientific_operations_false', all(value is False for value in manifest_readback['scientific_boundary'].values())),
])

failed_readback = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback:
    raise RuntimeError('Cell 7B5 readback verification failed:\n- ' + '\n- '.join(failed_readback))

immutable_hashes_after = OrderedDict((key, sha256_file(path)) for key, path in immutable_paths.items())
if immutable_hashes_after != immutable_hashes_before:
    raise AssertionError('One or more frozen upstream artifacts changed during Cell 7B5.')

total_checks = len(prewrite_checks) + len(readback_checks)
passed_checks = total_checks

separator = '=' * 152
print('\n' + separator)
print('EXPERIMENT 2 — STAGE 7B — CELL 7B5')
print('STAGE 7C EMBEDDING AND SEMANTIC RETRIEVAL EXECUTION AUTHORIZATION FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\nUPSTREAM CELL 7B4 REVERIFICATION')
print(f'Cell 7B4 manifest SHA-256                     : {sha256_file(cell_7b4_paths["manifest"])}')
print('Cell 7B4 terminal PASS verified               : YES')
print(f'Frozen Cell 7B4 artifacts                     : {len(cell_7b4_paths)}/8 exact hashes + sidecars')
print('Cell 7B4 QC failures                          : 0')

print('\nAUTHORIZED SCORE-BLIND EXECUTION INPUTS')
print(f'Semantic corpus SHA-256                       : {sha256_file(execution_input_paths["semantic_corpus"])}')
print('Semantic corpus expected rows                 : 100,920')
print(f'Primary questions SHA-256                     : {sha256_file(execution_input_paths["primary_questions"])}')
print('Primary questions                             : 80')
print('Answer-key outcomes inspected                 : NO')
print('Cell 7A3 scores loaded                        : NO')

print('\nCELL 7C0 AUTHORIZATION')
print('Embedding model                               : NeuML/pubmedbert-base-embeddings')
print('Embedding revision                            : b79526d6ef3645e0df4530322e266f24c829f5ef')
print('Embedding output                              : float32, L2-normalized, 768 dimensions')
print('Similarity/index                              : cosine via exact FAISS IndexFlatIP')
print('Semantic candidate pool                       : top-20 common pool for 80 questions')
print('Expected candidate rows                       : 1,600')
print('Quality reranking / RRF                       : NOT AUTHORIZED')
print('Final top-5 contexts                          : NOT AUTHORIZED')
print('Prompts / LLM                                 : NOT AUTHORIZED')
print('Answer keys / metrics                         : NOT AUTHORIZED')

print('\nCELL 7B5 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\nSCIENTIFIC OPERATIONS IN CELL 7B5')
print('Embeddings generated                          : NO')
print('FAISS index constructed                       : NO')
print('Semantic retrieval executed                   : NO')
print('Quality reranking applied                     : NO')
print('Prompts materialized                          : NO')
print('LLM called                                     : NO')
print('Adjudication or RAG metrics                    : NO')

print('\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C0                           : Embedding generation, exact FAISS indexing,')
print('                                                 and common semantic top-20 retrieval only')
print('Cell 7A3 score loading                        : PROHIBITED')
print('Quality reranking / top-5 / prompts / LLM     : PROHIBITED')
print(f'\nFINAL DECISION                                : {terminal_decision}')
print(separator)


EXPERIMENT 2 — STAGE 7B — CELL 7B5
STAGE 7C EMBEDDING AND SEMANTIC RETRIEVAL EXECUTION AUTHORIZATION FREEZE
Notebook                                      : 06_GES_Aware_Genomic_RAG_Cell_7B5_Stage_7C_Execution_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM CELL 7B4 REVERIFICATION
Cell 7B4 manifest SHA-256                     : 18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe
Cell 7B4 terminal PASS verified               : YES
Frozen Cell 7B4 artifacts                     : 8/8 exact hashes + sidecars
Cell 7B4 QC failures                          : 0

AUTHORIZED SCORE-BLIND EXECUTION INPUTS
Semantic corpus SHA-256                       : 2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399
Semantic corpus expected rows                 : 100,920
Primary questions SHA-256                     : c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df
Primary questions           